# 03 — Silver RENAMU
## Transformación Bronze → Silver

Procesa el **Registro Nacional de Municipalidades (RENAMU)** y enriquece los datos con
la categoría municipal desde el archivo CSV estático `categorias.csv`.

**Fuente:** `data/bronze/renamu/2022/batch_*.json`  
**Destino:** `data/silver/renamu/` (Parquet plano)

**Transformaciones:**
- Limpieza de ghost nulls
- Normalización de strings
- Join con `data/static/categorias.csv` para agregar `CATEGORIA` municipal


In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F

from src.paths import BRONZE, SILVER, CATEGORIAS_CSV, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from src.transformations.common import clean_ghost_nulls
from core.audit.control_manager import ControlManager, ExecutionStatus

print("✅ Imports OK")

In [ ]:
BRONZE_PATH = str_path(BRONZE["renamu"])
SILVER_PATH = str_path(SILVER["renamu"])
CATEGORIAS_PATH = str_path(CATEGORIAS_CSV)

print(f"Bronze source     : {BRONZE_PATH}")
print(f"Silver target     : {SILVER_PATH}")
print(f"Categorías CSV    : {CATEGORIAS_PATH}")
print(f"CSV existe        : {CATEGORIAS_CSV.exists()}")

ensure_dirs()

In [ ]:
spark = get_spark(app_name="MEF_Silver_RENAMU", memory="4g")

## PARTE 1: Lectura y Limpieza Bronze

In [ ]:
from functools import reduce

bronze_base = BRONZE["renamu"]
if not bronze_base.exists() or not any(bronze_base.rglob("*.json")):
    raise FileNotFoundError(f"Sin datos Bronze RENAMU en {bronze_base}. Ejecutar: python -m bronze.ingest_bronze")

dfs = []
for year_dir in sorted([d for d in bronze_base.iterdir() if d.is_dir()]):
    year = year_dir.name
    if not list(year_dir.rglob("*.json")):
        continue
    year_df = spark.read.option("multiline", "false").json(str_path(year_dir))
    year_df = year_df.withColumn("ANO_RENAMU", F.lit(year))
    dfs.append(year_df)

if not dfs:
    raise ValueError("No se encontraron datos JSON de RENAMU.")

raw_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)

n_raw = raw_df.count()
print(f"📦 Registros Bronze RENAMU: {n_raw:,}")
print(f"   Columnas totales: {len(raw_df.columns)}")

# Limpieza de ghost nulls
clean_df = clean_ghost_nulls(raw_df)
print("✅ Ghost nulls limpiados")

## PARTE 2: Enriquecimiento con Categorías Municipales

El archivo `categorias.csv` contiene el cruce exacto mediante el UBIGEO
y su categoría (ej: A, B, C, D o MANCOMUNIDAD).

Se realiza el join directo usando el código de ubigeo de 6 dígitos.


In [ ]:
# Encontrar la columna de ubigeo en RENAMU
ubigeo_col = next((c for c in clean_df.columns if c.upper() == "UBIGEO"), None)
print(f"Columna de UBIGEO encontrada: {ubigeo_col}")

if ubigeo_col and CATEGORIAS_CSV.exists():
    # Leer CSV de categorías
    cat_df = spark.read.option("header", "true").csv(CATEGORIAS_PATH)
    print(f"📂 Categorías cargadas: {cat_df.count()} entradas")

    # Corregir UBIGEO en cat_df (completar ceros a la izquierda a 6 dígitos)
    cat_df = cat_df.withColumn("UBIGEO_REF", F.lpad(F.col("UBIGEO").cast("string"), 6, "0"))
    
    # Seleccionar solo las columnas necesarias y quitar duplicados por UBIGEO
    cat_df = cat_df.select("UBIGEO_REF", "CATEGORIA").dropDuplicates(["UBIGEO_REF"])

    # Join con el catálogo de categorías por UBIGEO
    clean_df = clean_df.join(cat_df, F.col(ubigeo_col) == cat_df.UBIGEO_REF, "left")
    clean_df = clean_df.fillna({"CATEGORIA": "SIN_CATEGORIA"})
    clean_df = clean_df.drop("UBIGEO_REF", "UBIGEO_csv")

    # Resumen del cruce
    n_con_cat = clean_df.filter(F.col("CATEGORIA") != "SIN_CATEGORIA").count()
    print(f"✅ Join completado por UBIGEO: {n_con_cat:,}/{n_raw:,} con categoría ({n_con_cat/n_raw*100:.1f}%)")
    clean_df.groupBy("CATEGORIA").count().orderBy("count", ascending=False).show()
else:
    print("⚠️  Sin columna de UBIGEO o CSV no encontrado — agregando CATEGORIA='SIN_CATEGORIA'")
    clean_df = clean_df.withColumn("CATEGORIA", F.lit("SIN_CATEGORIA"))


## PARTE 3: Escritura a Silver

In [ ]:
control = ControlManager(pipeline_name="silver_renamu")
execution = control.start(input_parameters={"source": "renamu"})

start_time = time.time()
try:
    n_final = write_parquet(clean_df, SILVER_PATH, mode="overwrite", partition_by=["ANO_RENAMU"])
    elapsed = time.time() - start_time

    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={"raw_rows": n_raw, "clean_rows": n_final, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\n🎉 Silver RENAMU completado en {elapsed:.1f}s")
    print(f"   {n_raw:,} raw → {n_final:,} Silver")
except Exception as e:
    control.log_error("SilverRenamuError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

In [ ]:
# ── Verificación final ─────────────────────────────────────────────────────────
df_check = spark.read.parquet(SILVER_PATH)
print(f"📋 Verificación Silver RENAMU:")
print(f"  Total filas  : {df_check.count():,}")
print(f"  Total columnas: {len(df_check.columns)}")
df_check.limit(3).show(truncate=True)